# Chapter 14 — Frontiers: Where Mathematics Meets Research

**Math for ML Engineers** · companion notebook

This notebook mirrors §14.2 (natural gradient and the Fisher information matrix), §14.3 (score matching), and §14.4 (LoRA) of the chapter, implementing the second-order geometry and parameter-efficient fine-tuning ideas that sit at the edge of what the rest of the book builds toward. **Key takeaway: the same Fisher information matrix that defines the "correct" geometry for gradient descent (natural gradient) is, in one form or another, behind K-FAC, Adam's preconditioning, and the entire practice of approximating expensive second-order curvature at scale — while LoRA sidesteps curvature entirely by exploiting the fact that fine-tuning updates are empirically low-rank.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(14)

## Setup: the Fisher information matrix and natural gradient

For a model $p(y \mid x; \theta)$, the **Fisher information matrix** measures the local curvature of the log-likelihood surface — how sensitive the distribution is to small changes in $\theta$:

$$
F(\theta) = \mathbb{E}_{y \sim p(y|x;\theta)}\left[\nabla_\theta \log p(y|x;\theta)\, \nabla_\theta \log p(y|x;\theta)^\top\right]
$$

Vanilla gradient descent, $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$, takes the steepest-descent direction under the *Euclidean* metric on parameter space. But parameter space is not the object we actually care about — the space of *distributions* $p(y|x;\theta)$ is, and Euclidean distance between two $\theta$ vectors can be a wildly misleading proxy for how different the resulting distributions actually are. The **natural gradient** (§14.2) corrects for this by taking the steepest-descent direction under the KL-divergence geometry, for which $F$ is exactly the local Riemannian metric:

$$
\theta \leftarrow \theta - \eta\, F(\theta)^{-1} \nabla_\theta \mathcal{L}(\theta)
$$

For a softmax classifier with logits $z$ and $p = \operatorname{softmax}(z)$, the score $\nabla_z \log p(y) = \mathbb{1}[y] - p$, so $F$ can be estimated empirically by sampling $y \sim p$ and averaging the outer product of that score with itself — no analytic Hessian required.

In [ ]:
def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


def fisher_categorical(logits, n_samples=2000, rng=None):
    """Empirical Fisher for categorical p(y|x;theta)=softmax(logits). F ~ E[grad log p . grad log p^T]."""
    if rng is None: rng = np.random.default_rng(0)
    K = len(logits)
    p = softmax(logits)
    F = np.zeros((K, K))
    ys = rng.choice(K, size=n_samples, p=p)
    for y in ys:
        onehot = np.zeros(K); onehot[y] = 1.0
        g = onehot - p
        F += np.outer(g, g)
    return F / n_samples


def natural_gradient_step(grad, F, lr, eps=1e-4):
    """theta <- theta - lr * F^-1 g (natural gradient = steepest descent in KL geometry). eps*I damping."""
    K = len(grad)
    F_reg = F + eps * np.eye(K)
    return -lr * np.linalg.solve(F_reg, grad)


# Toy 3-class softmax classifier: start with near-uniform logits, push toward class 0
TARGET_CLASS = 0
logits0 = np.array([0.1, 0.0, -0.1])
print(f"Initial logits: {logits0}, initial p(y): {softmax(logits0)}")

## Vanilla vs. natural gradient on a toy softmax classifier

We track a single scalar objective: push $p(y = \text{target})$ toward 1 by descending the negative log-likelihood $\mathcal{L}(z) = -\log p(\text{target} \mid z)$, whose gradient w.r.t. logits is $\nabla_z \mathcal{L} = p - \mathbb{1}[\text{target}]$ (the same score vector, negated, that builds $F$ above).

At each step we compute the gradient, estimate the empirical Fisher $F$ at the *current* logits via `fisher_categorical`, and take two parallel updates from the same starting point: a vanilla step $z \leftarrow z - \eta \nabla_z \mathcal{L}$, and a natural-gradient step $z \leftarrow z - \eta F^{-1} \nabla_z \mathcal{L}$ via `natural_gradient_step`. Both use the same learning rate, so any difference in convergence speed comes entirely from the curvature correction $F^{-1}$ applies — not from a larger step.

In [ ]:
def run_gradient_descent(logits_init, target, n_steps, lr, natural=False, rng=None):
    if rng is None:
        rng = np.random.default_rng(14)
    z = logits_init.copy()
    K = len(z)
    onehot = np.zeros(K); onehot[target] = 1.0
    p_history = [softmax(z)[target]]

    for _ in range(n_steps):
        p = softmax(z)
        grad = p - onehot  # d(-log p[target]) / dz

        if natural:
            F = fisher_categorical(z, n_samples=2000, rng=rng)
            step = natural_gradient_step(grad, F, lr)
        else:
            step = -lr * grad

        z = z + step
        p_history.append(softmax(z)[target])

    return np.array(p_history)


N_STEPS = 25
LR = 0.5

rng_vanilla = np.random.default_rng(14)
rng_natural = np.random.default_rng(14)

p_vanilla = run_gradient_descent(logits0, TARGET_CLASS, N_STEPS, LR, natural=False, rng=rng_vanilla)
p_natural = run_gradient_descent(logits0, TARGET_CLASS, N_STEPS, LR, natural=True, rng=rng_natural)

print(f"p(target) after {N_STEPS} steps -- vanilla gradient: {p_vanilla[-1]:.4f}")
print(f"p(target) after {N_STEPS} steps -- natural gradient: {p_natural[-1]:.4f}")
print(f"Steps for vanilla to first exceed 0.9: "
      f"{np.argmax(p_vanilla > 0.9) if (p_vanilla > 0.9).any() else 'never'}")
print(f"Steps for natural to first exceed 0.9: "
      f"{np.argmax(p_natural > 0.9) if (p_natural > 0.9).any() else 'never'}")

## Visualizing convergence, and the LoRA parameter cliff

Two plots make both stories in this chapter legible at a glance. First, $p(y = \text{target})$ against optimization step for both gradient methods on identical initial conditions and learning rate: the natural-gradient trajectory should rise faster and straighter toward 1, while vanilla gradient takes a shallower, slower path because it is implicitly fighting the mismatch between Euclidean and KL geometry near the softmax's saturated regions. Second, a log-scale bar chart of `lora_params`' full-vs-LoRA trainable-parameter counts across the three $(d, r)$ configurations from the source module, annotated with the compression percentage each configuration achieves — the log scale is necessary because full fine-tuning's parameter count is orders of magnitude larger than LoRA's at realistic model widths.

In [ ]:
def lora_params(d, r):
    """LoRA: W = W0 + BA, B in R^(dxr), A in R^(rxd). Trainable = 2dr vs full d^2."""
    full   = d * d
    lora   = 2 * d * r
    saving = 1 - lora / full
    return {"full": full, "lora": lora, "compression": f"{saving:.1%}"}


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: natural vs. vanilla gradient trajectory
steps = np.arange(N_STEPS + 1)
axes[0].plot(steps, p_vanilla, "o-", color="C0", label="vanilla gradient", markersize=4)
axes[0].plot(steps, p_natural, "s-", color="C3", label="natural gradient", markersize=4)
axes[0].axhline(1.0, color="gray", linestyle=":", linewidth=1)
axes[0].set_xlabel("optimization step")
axes[0].set_ylabel(r"$p(y = \mathrm{target})$")
axes[0].set_title("Natural vs. vanilla gradient (3-class softmax)")
axes[0].legend(loc="lower right")

# Right: LoRA vs full parameter counts, log scale, annotated with compression %
configs = [(768, 8), (4096, 8), (4096, 64)]
labels = [f"d={d}\nr={r}" for d, r in configs]
full_counts = [lora_params(d, r)["full"] for d, r in configs]
lora_counts = [lora_params(d, r)["lora"] for d, r in configs]
compressions = [lora_params(d, r)["compression"] for d, r in configs]

x = np.arange(len(configs))
width = 0.35
axes[1].bar(x - width / 2, full_counts, width, label="full fine-tune ($d^2$)", color="C0")
axes[1].bar(x + width / 2, lora_counts, width, label="LoRA ($2dr$)", color="C3")
axes[1].set_yscale("log")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel("trainable parameters (log scale)")
axes[1].set_title("LoRA vs. full fine-tuning parameter count")
axes[1].legend(loc="upper left")

for xi, lc, comp in zip(x, lora_counts, compressions):
    axes[1].annotate(comp, (xi + width / 2, lc), textcoords="offset points",
                      xytext=(0, 8), ha="center", fontsize=9, color="C3")

plt.tight_layout()
plt.show()

steps_to_90_vanilla = np.argmax(p_vanilla > 0.9) if (p_vanilla > 0.9).any() else N_STEPS
steps_to_90_natural = np.argmax(p_natural > 0.9) if (p_natural > 0.9).any() else N_STEPS
print(f"Steps to reach p(target) > 0.9 -- vanilla: {steps_to_90_vanilla}, natural: {steps_to_90_natural}")
print(f"Natural gradient reaches the target probability in "
      f"{steps_to_90_vanilla - steps_to_90_natural} fewer steps at the same learning rate, "
      "because F^-1 rescales the update to account for how flat/curved the log-likelihood "
      "surface is in each direction -- exactly the correction Euclidean gradient descent lacks.")

for (d, r), comp in zip(configs, compressions):
    print(f"d={d:>5}, r={r:>3}: LoRA saves {comp} of parameters vs. full fine-tuning")

## Why this matters at production scale

The natural gradient's $F^{-1}$ correction is mathematically exact but computationally brutal: for a model with $P$ parameters, $F$ is a $P \times P$ matrix, and inverting it costs $O(P^3)$. For a billion-parameter language model this is not merely expensive, it is impossible to even store. This is precisely the gap that **K-FAC** (Kronecker-Factored Approximate Curvature) and related second-order optimizers fill in production: K-FAC approximates $F$ as block-diagonal across layers, and further factors each per-layer block as a Kronecker product of two much smaller matrices, making an approximate natural-gradient step tractable at scale. Every "structured" or "block-diagonal" curvature approximation you'll encounter in large-scale training research is a variant of this same idea — the true Fisher is intractable, so approximate its structure instead of its every entry.

There is a second, quieter appearance of this same idea that touches nearly every training run you'll ever launch: **Adam's per-parameter second-moment estimate $\hat{v}_t$ is itself a crude diagonal approximation to Fisher preconditioning.** Adam divides each gradient coordinate by $\sqrt{\hat{v}_t} + \epsilon$ — a running estimate of that coordinate's squared gradient magnitude — which is exactly a diagonal (all off-diagonal curvature ignored) empirical-Fisher preconditioner. This is why Adam so reliably outperforms plain SGD on ill-conditioned loss landscapes like transformer training: it is doing a cheap, diagonal-only version of the natural gradient step derived above, for the cost of one extra buffer per parameter instead of a full $F^{-1}$ solve.

**LoRA** (Low-Rank Adaptation) sidesteps the curvature problem in a completely different way, and is today the dominant method for fine-tuning large language models in practice. Instead of updating the full weight matrix $W \in \mathbb{R}^{d \times d}$, LoRA freezes $W_0$ and learns a low-rank update $W = W_0 + BA$ with $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$, training only $2dr$ parameters instead of $d^2$ (§14.4). For a realistic $d = 4096$ transformer weight matrix at rank $r = 8$, that's roughly $2 \times 4096 \times 8 \approx 65{,}500$ trainable parameters against $4096^2 \approx 16.8$ million for full fine-tuning — **about 0.4% of the parameters**, which is why LoRA (and its quantized variant QLoRA) makes it possible to fine-tune multi-billion-parameter models on a single consumer GPU.

In [ ]:
print(f"{'d':>6} {'r':>4} {'full params':>14} {'LoRA params':>13} {'compression':>12}")
for d, r in configs:
    stats = lora_params(d, r)
    print(f"{d:>6} {r:>4} {stats['full']:>14,} {stats['lora']:>13,} {stats['compression']:>12}")

# Cross-check the headline claim from the insight cell: rank-8 LoRA on a 4096-dim
# matrix trains roughly what fraction of full fine-tuning's parameters?
d4096_stats = lora_params(4096, 8)
frac_of_full = d4096_stats["lora"] / d4096_stats["full"]
print(f"\nAt d=4096, r=8: LoRA trains {frac_of_full:.4%} of the full fine-tuning parameter count "
      f"({d4096_stats['lora']:,} vs {d4096_stats['full']:,}).")

# Also confirm Adam's diagonal preconditioning matches a DIAGONAL slice of the
# empirical Fisher computed earlier -- i.e. Adam ~ diag(F)^{-1}, not full F^{-1}.
F_full = fisher_categorical(logits0, n_samples=5000, rng=np.random.default_rng(14))
F_diag_only = np.diag(np.diag(F_full))
diag_vs_full_offdiag_mass = np.sum(np.abs(F_full - F_diag_only)) / np.sum(np.abs(F_full))
print(f"\nFraction of empirical Fisher's mass sitting OFF the diagonal: "
      f"{diag_vs_full_offdiag_mass:.1%}")
print("That off-diagonal mass is exactly the curvature information Adam's diagonal "
      "second-moment estimate discards -- the price paid for a cheap, per-parameter-only "
      "preconditioner instead of a full natural-gradient solve.")

## Exercise

1. `fisher_categorical` estimates $F$ empirically by sampling $y \sim p$. Derive the **exact, closed-form** Fisher information matrix for a softmax categorical distribution (it has a clean expression in terms of $p$ alone — no sampling needed) and confirm numerically that `fisher_categorical` converges to it as `n_samples` grows. §2.7 (Fisher information) and §2.3.2 (the exponential family) give the general formula this specializes.

2. Extend `natural_gradient_step` to a **block-diagonal** Fisher approximation in the spirit of K-FAC: partition a larger parameter vector into blocks, compute (or approximate) $F$ only within each block, and zero out cross-block terms before inverting. Compare the resulting trajectory's convergence speed against both the full natural-gradient and vanilla-gradient trajectories from this notebook, and discuss what accuracy is lost by discarding the cross-block curvature. §5.7.4 (diagonal and block curvature) is the relevant background for justifying when this is a safe simplification.